In [2]:
import os
import numpy as np
import pandas as pd
from scipy import stats

from dotenv import load_dotenv
load_dotenv()

True

In [3]:
RESULTS_DIR_PATH = os.environ.get('RESULTS_FOLDER')

COMPETITION_RESULTS_CSV_PATH = '.' + os.environ.get('RESULTS_FOLDER') + 'prod/competition_full/competition/results.csv'
CHOICE_RESULTS_CSV_PATH = '.' + os.environ.get('RESULTS_FOLDER') + 'prod/choice_full/choice/results.csv'
FAMILIARITY_CHOICE_CSV_PATH = '.' + os.environ.get('RESULTS_FOLDER') + 'prod/familiarity_choice_full/familiarity_choice/results.csv'

#### Functions

In [33]:
def _round_or_nan(value, digits: int):
    return round(float(value), digits) if pd.notna(value) and np.isfinite(value) else np.nan

def _to_number_array(values: pd.Series) -> np.ndarray:
    x = pd.to_numeric(values, errors='coerce')
    x = x[np.isfinite(x)]
    return x.to_numpy(dtype=float)

def _to_boolean_array(values: pd.Series) -> np.ndarray:
    x = pd.to_numeric(values, errors='coerce')
    x = x[np.isfinite(x)]

    arr = x.to_numpy(dtype=float)

    if len(arr) == 0:
        return arr.astype(int)

    unique_values = set(np.unique(arr))
    
    if not unique_values.issubset({0.0, 1.0}):
        raise RuntimeError(f'Boolean metric must contain only 0/1. Found: {unique_values}')

    return arr.astype(int)

def _summary(values: pd.Series, metric_type: str):
    if metric_type == 'boolean':
        arr = _to_boolean_array(values)
    else:
        arr = _to_number_array(values)

    n = len(arr)
    avg = float(np.mean(arr)) if n > 0 else np.nan
    std = float(np.std(arr, ddof=1)) if n > 1 else np.nan

    return arr, n, avg, std

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [34]:
def _two_prop_z_test(
    test: np.ndarray,
    control: np.ndarray,
    alternative: str = "two-sided",
):
    n_test = len(test)
    n_control = len(control)

    x_test = int(np.sum(test))
    x_control = int(np.sum(control))

    p_test = x_test / n_test
    p_control = x_control / n_control

    p_pool = (x_test + x_control) / (n_test + n_control)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_test + 1 / n_control))

    if se == 0:
        return 0.0, 1.0

    z_stat = (p_test - p_control) / se

    if alternative == "two-sided":
        p_value = 2 * stats.norm.sf(abs(z_stat))
    elif alternative == "greater":
        p_value = stats.norm.sf(z_stat)
    elif alternative == "less":
        p_value = stats.norm.cdf(z_stat)
    else:
        raise RuntimeError("alternative must be 'two-sided', 'greater', or 'less'")

    return float(z_stat), float(p_value)

In [35]:
def _boolean_significance(
    control: np.ndarray,
    test: np.ndarray,
    *,
    alternative: str,
    min_n_for_z: int,
    min_expected_for_z: float,
):
    n_control = len(control)
    n_test = len(test)

    if n_control == 0 or n_test == 0:
        return "not_enough_data", np.nan, np.nan

    x_control = int(np.sum(control))
    x_test = int(np.sum(test))

    # rows: test/control
    # columns: success/failure
    table = np.array([
        [x_test, n_test - x_test],
        [x_control, n_control - x_control],
    ])

    expected = stats.contingency.expected_freq(table)

    use_fisher = (
        min(n_control, n_test) < min_n_for_z
        or np.any(expected < min_expected_for_z)
    )

    if use_fisher:
        res = stats.fisher_exact(table, alternative=alternative)
        return "fisher_exact", float(res.statistic), float(res.pvalue)

    z_stat, p_value = _two_prop_z_test(
        test=test,
        control=control,
        alternative=alternative,
    )

    return "two_proportion_z_test", z_stat, p_value

In [36]:
def _permutation_mean_test(
    control: np.ndarray,
    test: np.ndarray,
    *,
    alternative: str,
    n_resamples: int,
    random_state: int,
):
    def statistic(test_sample, control_sample):
        return np.mean(test_sample) - np.mean(control_sample)

    n_test = len(test)
    n_control = len(control)

    total_combinations = math.comb(n_test + n_control, n_test)
    actual_resamples = np.inf if total_combinations <= n_resamples else n_resamples

    try:
        res = stats.permutation_test(
            data=(test, control),
            statistic=statistic,
            permutation_type="independent",
            alternative=alternative,
            n_resamples=actual_resamples,
            vectorized=False,
            rng=np.random.default_rng(random_state),
        )
    except TypeError:
        # Для старых версий SciPy
        res = stats.permutation_test(
            data=(test, control),
            statistic=statistic,
            permutation_type="independent",
            alternative=alternative,
            n_resamples=actual_resamples,
            vectorized=False,
            random_state=random_state,
        )

    return "permutation_mean_test", float(res.statistic), float(res.pvalue)

In [37]:
def _number_significance(
    control: np.ndarray,
    test: np.ndarray,
    *,
    alternative: str,
    small_n_threshold: int,
    n_resamples: int,
    random_state: int,
):
    if len(control) < 2 or len(test) < 2:
        return "not_enough_data", np.nan, np.nan

    if min(len(control), len(test)) < small_n_threshold:
        return _permutation_mean_test(
            control=control,
            test=test,
            alternative=alternative,
            n_resamples=n_resamples,
            random_state=random_state,
        )

    res = stats.ttest_ind(
        test,
        control,
        equal_var=False,
        alternative=alternative,
        nan_policy="omit",
    )

    return "welch_t_test", float(res.statistic), float(res.pvalue)

In [47]:
def group_function(
    group_df: pd.DataFrame,
    condition_column: str,
    condition_control: str,
    metric_columns_types: dict[str, str],
    small_n_number: int = 20,
    min_n_for_z_boolean: int = 30,
    min_expected_for_z_boolean: float = 5.0,
    n_resamples: int = 20000,
    random_state: int = 42
) -> pd.Series:
    if condition_column not in group_df.columns:
        raise RuntimeError(f'Column {condition_column} is not found in group DataFrame.')

    if group_df[group_df[condition_column] == condition_control].empty:
        raise RuntimeError(f'Condition {condition_control} is not found in {condition_column} column.')

    for metric_column in metric_columns_types.keys():
        if metric_column not in group_df.columns:
            raise RuntimeError(f'Metric column {metric} is not found in group DataFrame.')

    result = {}

    control_df = group_df[group_df[condition_column] == condition_control]
    control_stats = {}

    for metric_column, metric_type in metric_columns_types.items():
        control_arr, control_n, control_avg, control_std = _summary(control_df[metric_column], metric_type)

        control_stats[metric_column] = {'arr': control_arr, 'n': control_n, 'avg': control_avg, 'std': control_std, 'type': metric_type}

        digits = 3 if metric_type == 'boolean' else 1

        result[(metric_column, condition_control, 'N')] = int(control_n)
        result[(metric_column, condition_control, 'Avg')] = _round_or_nan(control_avg, digits)
        result[(metric_column, condition_control, 'Std')] = _round_or_nan(control_std, digits)

    for condition in group_df[condition_column].dropna().unique():
        if condition == condition_control:
            continue

        condition_df = group_df[group_df[condition_column] == condition]

        for metric_column, metric_type in metric_columns_types.items():
            test_arr, test_n, test_avg, test_std = _summary(condition_df[metric_column], metric_type)

            control_arr = control_stats[metric_column]['arr']
            control_avg = control_stats[metric_column]['avg']

            digits = 3 if metric_type == 'boolean' else 1

            result[(metric_column, condition, 'N')] = int(test_n)
            result[(metric_column, condition, 'Avg')] = _round_or_nan(test_avg, digits)
            result[(metric_column, condition, 'Std')] = _round_or_nan(test_std, digits)

            diff = test_avg - control_avg if pd.notna(test_avg) and pd.notna(control_avg) else np.nan
            r_diff = 100 * (test_avg / control_avg - 1) if pd.notna(test_avg) and pd.notna(control_avg) and control_avg != 0 else np.nan

            result[(metric_column, condition, 'Diff')] = _round_or_nan(diff, digits)
            result[(metric_column, condition, 'R. Diff')] = _round_or_nan(r_diff, 1)

            if metric_type == 'boolean':
                test_name, stat_value, p_value = _boolean_significance(
                    control=control_arr,
                    test=test_arr,
                    alternative='two-sided',
                    min_n_for_z=min_n_for_z_boolean,
                    min_expected_for_z=min_expected_for_z_boolean,
                )
            else:
                test_name, stat_value, p_value = _number_significance(
                    control=control_arr,
                    test=test_arr,
                    alternative='two-sided',
                    small_n_threshold=small_n_number,
                    n_resamples=n_resamples,
                    random_state=random_state,
                )

            result[(metric_column, condition, 'P-Value')] = _round_or_nan(p_value, 4)

    return pd.Series(result)

In [58]:
def _sort_metric_condition_stat_columns(
    df: pd.DataFrame,
    condition_control: str,
) -> pd.DataFrame:
    if not isinstance(df.columns, pd.MultiIndex):
        return df

    stats_order_control = ['N', 'Avg', 'Std']
    stats_order_test = ['N', 'Avg', 'Std', 'Diff', 'R. Diff', 'P-Value']

    metrics = list(dict.fromkeys(df.columns.get_level_values(0)))
    conditions = list(dict.fromkeys(df.columns.get_level_values(1)))

    ordered_cols = []

    for metric in metrics:
        if condition_control in conditions:
            for stat in stats_order_control:
                col = (metric, condition_control, stat)
                if col in df.columns:
                    ordered_cols.append(col)

        for condition in conditions:
            if condition == condition_control:
                continue
            for stat in stats_order_test:
                col = (metric, condition, stat)
                if col in df.columns:
                    ordered_cols.append(col)

    remaining = [col for col in df.columns if col not in ordered_cols]
    ordered_cols.extend(remaining)

    return df.loc[:, ordered_cols]

In [65]:
def calc(
    df: pd.DataFrame,
    group_columns: list[str],
    condition_column: str,
    condition_control: str,
    metric_columns_types: dict[str, str],
    small_n_number: int = 20,
    min_n_for_z_boolean: int = 30,
    min_expected_for_z_boolean: float = 5.0,
    n_resamples: int = 20000,
    random_state: int = 42
) -> pd.DataFrame:
    for group_column in group_columns:
        if group_column not in df.columns:
            raise RuntimeError(f'{group_column} is not found in DataFrame.')
    
    result = df.groupby(by=group_columns).apply(
        lambda g_df: group_function(
            group_df=g_df,
            condition_column=condition_column,
            condition_control=condition_control,
            metric_columns_types=metric_columns_types,
            small_n_number=small_n_number,
            min_n_for_z_boolean=min_n_for_z_boolean,
            min_expected_for_z_boolean=min_expected_for_z_boolean,
            n_resamples=n_resamples,
            random_state=random_state),
        include_groups=False
    )
    
    result.columns = pd.MultiIndex.from_tuples(result.columns)
    result = _sort_metric_condition_stat_columns(result, condition_control=condition_control)

    return result

## Experiment 1. Competition

#### Data

In [8]:
df = pd.read_csv(FAMILIARITY_CHOICE_CSV_PATH)

In [9]:
df.isna().sum()

gender                              0
model_id                            0
would_buy_lottery_ticket            0
chosen_ticket                    1250
would_exchange_lottery_ticket       0
experiment_id                       0
variation_id                        0
condition_id                        0
iteration_id                        0
dtype: int64

In [87]:
df.groupby(by=['experiment_id', 'variation_id', 'model_id']).count()

gender  \
experiment_id variation_id   model_id                      
choice        default        deepseek_v3.2           216   
                             gpt_5                   216   
                             gpt_5_mini              216   
                             gpt_5_nano              216   
                             grok_4.20               216   
                             grok_4_fast             216   
                             qwen_3_max_thinking     216   
                             qwen_turbo              216   
              fully_rational deepseek_v3.2           216   
                             gpt_5                   216   
                             gpt_5_mini              216   
                             gpt_5_nano              216   
                             grok_4.20               216   
                             grok_4_fast             216   
                             qwen_3_max_thinking     216   
                             qwen_turbo              216   
              large_pot      deepseek_v3.2           216   
                             gpt_5                   216   
                             gpt_5_mini              216   
                             gpt_5_nano              216   
                             grok_4.20               216   
                             grok_4_fast             216   
                             qwen_3_max_thinking     216   
                             qwen_turbo              216   
              letter_ticket  deepseek_v3.2           216   
                             gpt_5                   216   
                             gpt_5_mini              216   
                             gpt_5_nano              216   
                             grok_4.20               216   
                             grok_4_fast             216   
                             qwen_3_max_thinking     216   
                             qwen_turbo              216   
              rephrasing     deepseek_v3.2           216   
                             gpt_5                   216   
                             gpt_5_mini              216   
                             gpt_5_nano              216   
                             grok_4.20               216   
                             grok_4_fast             216   
                             qwen_3_max_thinking     216   
                             qwen_turbo              216   

                                                  would_buy_lottery_ticket  \
experiment_id variation_id   model_id                                        
choice        default        deepseek_v3.2                             216   
                             gpt_5                                     216   
                             gpt_5_mini                                216   
                             gpt_5_nano                                216   
                             grok_4.20                                 216   
                             grok_4_fast                               216   
                             qwen_3_max_thinking                       216   
                             qwen_turbo                                216   
              fully_rational deepseek_v3.2                             216   
                             gpt_5                                     216   
                             gpt_5_mini                                216   
                             gpt_5_nano                                216   
                             grok_4.20                                 216   
                             grok_4_fast                               216   
                             qwen_3_max_thinking                       216   
                             qwen_turbo                                216   
              large_pot      deepseek_v3.2                             216   
                             gpt_5             

In [75]:
df.head()

,gender,model_id,bet_1,bet_2,bet_3,bet_4,competence,experiment_id,variation_id,condition_id,iteration_id
0,male,gpt_5_nano,0,0,0.0,0.0,6,competition,rephrasing,dapper,0
1,male,gpt_5_nano,10,10,10.0,10.0,6,competition,rephrasing,dapper,0
2,male,gpt_5_nano,15,12,9.0,6.0,6,competition,rephrasing,dapper,0
3,male,gpt_5_nano,10,15,20.0,25.0,5,competition,rephrasing,dapper,0
4,male,gpt_5_nano,10,15,20.0,25.0,5,competition,rephrasing,dapper,0


In [68]:
df['bet_avg'] = 0.25 * (df['bet_1'] + df['bet_2'] + df['bet_3'] + df['bet_4'])

#### Analytics

In [73]:
calc(
    df=df,
    group_columns=['experiment_id', 'variation_id', 'model_id'],
    condition_column='condition_id',
    condition_control='dapper',
    metric_columns_types=dict(competence='number'),
)

/Users/arsenosipyan/Desktop/ioc/venv/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
/Users/arsenosipyan/Desktop/ioc/venv/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
/Users/arsenosipyan/Desktop/ioc/venv/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
/Users/arsenosipyan/Desktop/ioc/venv/lib/python3.

competence            \
                                                       dapper             
                                                            N  Avg  Std   
experiment_id variation_id     model_id                                   
competition   default          deepseek_v3.2             54.0  5.6  0.9   
                               gpt_5                     54.0  6.0  0.0   
                               gpt_5_mini                54.0  6.0  0.0   
                               gpt_5_nano                54.0  5.7  0.5   
                               qwen_3_max_thinking       54.0  5.0  0.0   
                               qwen_turbo                54.0  6.0  0.0   
              fully_rational   deepseek_v3.2             54.0  5.1  1.9   
                               gpt_5                     54.0  4.9  0.9   
                               gpt_5_mini                54.0  6.0  0.0   
                               gpt_5_nano                54.0  5.6  0.5   
                               qwen_3_max_thinking       54.0  6.0  0.0   
                               qwen_turbo                54.0  6.0  0.0   
              higher_emphasis  deepseek_v3.2             54.0  5.1  1.7   
                               gpt_5                     54.0  6.0  0.2   
                               gpt_5_mini                54.0  5.9  0.3   
                               gpt_5_nano                54.0  5.6  0.5   
                               qwen_3_max_thinking       54.0  5.0  0.0   
                               qwen_turbo                54.0  5.9  0.4   
              lower_emphasis   deepseek_v3.2             54.0  4.8  1.8   
                               gpt_5                     54.0  5.2  0.4   
                               gpt_5_mini                54.0  5.3  0.5   
                               gpt_5_nano                54.0  4.7  0.6   
                               qwen_3_max_thinking       54.0  4.0  0.0   
                               qwen_turbo                54.0  6.0  0.3   
              rephrasing       deepseek_v3.2             54.0  5.3  1.4   
                               gpt_5                     54.0  5.5  0.5   
                               gpt_5_mini                54.0  5.8  0.4   
                               gpt_5_nano                54.0  5.3  0.5   
                               qwen_3_max_thinking       54.0  5.0  0.0   
                               qwen_turbo                54.0  6.0  0.0   
              without_emphasis deepseek_v3.2             54.0  3.4  1.8   
                               gpt_5                     54.0  4.1  0.2   
                               gpt_5_mini                54.0  4.0  0.4   
                               gpt_5_nano                54.0  3.9  0.2   
                               qwen_3_max_thinking       54.0  4.0  0.2   
                               qwen_turbo                54.0  6.0  0.0   

                                                                           \
                                                   schnook                  
                                                         N  Avg  Std Diff   
experiment_id variation_id     model_id                                     
competition   default          deepseek_v3.2          54.0  2.9  1.6 -2.8   
                               gpt_5                  54.0  2.1  0.2 -3.9   
                               gpt_5_mini             54.0  2.0  0.0 -4.0   
                               gpt_5_nano             54.0  2.7  0.8 -3.0   
                               qwen_3_max_thinking    54.0  2.0  0.0 -3.0   
                               qwen_turbo             54.0  2.0  0.9 -4.0   
              fully_rational   deepseek_v3.2          54.0  3.1  2.3 -2.0   
                               gpt_5                  54.0  3.3  0.7 -1.7   
                               gpt_5_mini             54.0  2.2  0.8 -3.8   
                               gpt_5_nano        

In [71]:
calc(
    df=df,
    group_columns=['experiment_id', 'variation_id', 'model_id'],
    condition_column='condition_id',
    condition_control='dapper',
    metric_columns_types=dict(bet_1='number'),
)

/Users/arsenosipyan/Desktop/ioc/venv/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
/Users/arsenosipyan/Desktop/ioc/venv/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
/Users/arsenosipyan/Desktop/ioc/venv/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
/Users/arsenosipyan/Desktop/ioc/venv/lib/python3.

bet_1                     \
                                                   dapper            schnook   
                                                        N   Avg  Std       N   
experiment_id variation_id     model_id                                        
competition   default          deepseek_v3.2         54.0   7.3  8.3    54.0   
                               gpt_5                 54.0  16.1  3.6    54.0   
                               gpt_5_mini            54.0   9.0  3.4    54.0   
                               gpt_5_nano            54.0   7.7  5.9    54.0   
                               qwen_3_max_thinking   54.0  14.3  1.5    54.0   
                               qwen_turbo            54.0  25.0  0.0    54.0   
              fully_rational   deepseek_v3.2         54.0   2.3  7.3    54.0   
                               gpt_5                 54.0   0.0  0.0    54.0   
                               gpt_5_mini            54.0   4.8  9.8    54.0   
                               gpt_5_nano            54.0   1.8  5.6    54.0   
                               qwen_3_max_thinking   54.0  23.1  6.6    54.0   
                               qwen_turbo            54.0  25.0  0.0    54.0   
              higher_emphasis  deepseek_v3.2         54.0   7.0  9.2    54.0   
                               gpt_5                 54.0  16.2  4.4    54.0   
                               gpt_5_mini            54.0   6.7  4.3    54.0   
                               gpt_5_nano            54.0   6.9  5.7    54.0   
                               qwen_3_max_thinking   54.0  12.3  2.5    54.0   
                               qwen_turbo            54.0  23.9  4.0    54.0   
              lower_emphasis   deepseek_v3.2         54.0   6.2  8.2    54.0   
                               gpt_5                 54.0  16.0  3.9    54.0   
                               gpt_5_mini            54.0   9.8  2.4    54.0   
                               gpt_5_nano            54.0   8.9  6.3    54.0   
                               qwen_3_max_thinking   54.0  14.7  1.0    54.0   
                               qwen_turbo            54.0  24.7  2.0    54.0   
              rephrasing       deepseek_v3.2         54.0  14.0  8.6    54.0   
                               gpt_5                 54.0  16.3  4.6    54.0   
                               gpt_5_mini            54.0  11.0  4.2    54.0   
                               gpt_5_nano            54.0  10.5  4.6    54.0   
                               qwen_3_max_thinking   54.0  12.4  1.1    54.0   
                               qwen_turbo            54.0  25.0  0.0    54.0   
              without_emphasis deepseek_v3.2         54.0   8.8  8.7    54.0   
                               gpt_5                 54.0  16.3  3.6    54.0   
                               gpt_5_mini            54.0  12.1  3.6    54.0   
                               gpt_5_nano            54.0   9.2  5.4    54.0   
                               qwen_3_max_thinking   54.0  14.0  1.8    54.0   
                               qwen_turbo            54.0  25.0  0.0    54.0   

                                                                             \
                                                                              
                                                     Avg   Std Diff R. Diff   
experiment_id variation_id     model_id                                       
competition   default          deepseek_v3.2         9.6   9.6  2.3    31.1   
                               gpt_5                15.8   3.6 -0.3    -1.6   
                               gpt_5_mini           13.3   5.7  4.3    47.8   
                               gpt_5_nano            9.2   6.7  1.5    18.9   
                               qwen_3_max_thinking  15.0   0.0  0.7     5.1   
                               qwen_turbo           24.7   2.0 -0.3    -1.1   
              fully_rational   deepseek_v3.2         6.4  10.

In [72]:
calc(
    df=df,
    group_columns=['experiment_id', 'variation_id', 'model_id'],
    condition_column='condition_id',
    condition_control='dapper',
    metric_columns_types=dict(bet_avg='number'),
)

/Users/arsenosipyan/Desktop/ioc/venv/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
/Users/arsenosipyan/Desktop/ioc/venv/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
/Users/arsenosipyan/Desktop/ioc/venv/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
/Users/arsenosipyan/Desktop/ioc/venv/lib/python3.

bet_avg                     \
                                                    dapper            schnook   
                                                         N   Avg  Std       N   
experiment_id variation_id     model_id                                         
competition   default          deepseek_v3.2          54.0   9.2  7.8    54.0   
                               gpt_5                  54.0  19.3  2.9    54.0   
                               gpt_5_mini             54.0  15.5  2.9    54.0   
                               gpt_5_nano             54.0   8.1  5.9    54.0   
                               qwen_3_max_thinking    54.0  17.1  1.0    54.0   
                               qwen_turbo             54.0  25.0  0.0    54.0   
              fully_rational   deepseek_v3.2          54.0   3.5  7.7    54.0   
                               gpt_5                  54.0   0.0  0.0    54.0   
                               gpt_5_mini             54.0   5.0  9.9    54.0   
                               gpt_5_nano             54.0   2.6  5.6    54.0   
                               qwen_3_max_thinking    54.0  23.1  6.6    54.0   
                               qwen_turbo             54.0  25.0  0.0    54.0   
              higher_emphasis  deepseek_v3.2          54.0   7.6  8.3    54.0   
                               gpt_5                  54.0  19.4  3.0    54.0   
                               gpt_5_mini             54.0  12.9  3.9    54.0   
                               gpt_5_nano             54.0   7.6  6.3    54.0   
                               qwen_3_max_thinking    54.0  17.4  0.4    54.0   
                               qwen_turbo             54.0  23.9  4.0    54.0   
              lower_emphasis   deepseek_v3.2          54.0   9.1  8.4    54.0   
                               gpt_5                  54.0  18.7  2.9    54.0   
                               gpt_5_mini             54.0  15.7  2.9    54.0   
                               gpt_5_nano             54.0   8.4  5.5    54.0   
                               qwen_3_max_thinking    54.0  17.3  0.6    54.0   
                               qwen_turbo             54.0  24.7  2.0    54.0   
              rephrasing       deepseek_v3.2          54.0  14.4  6.6    54.0   
                               gpt_5                  54.0  19.5  2.9    54.0   
                               gpt_5_mini             54.0  15.4  4.8    54.0   
                               gpt_5_nano             54.0  10.0  4.5    54.0   
                               qwen_3_max_thinking    54.0  18.2  1.8    54.0   
                               qwen_turbo             54.0  25.0  0.0    54.0   
              without_emphasis deepseek_v3.2          54.0  11.0  7.7    54.0   
                               gpt_5                  54.0  17.3  2.1    54.0   
                               gpt_5_mini             54.0  15.9  3.0    54.0   
                               gpt_5_nano             54.0   9.5  5.5    54.0   
                               qwen_3_max_thinking    54.0  17.2  0.9    54.0   
                               qwen_turbo             54.0  25.0  0.0    54.0   

                                                                             \
                                                                              
                                                     Avg   Std Diff R. Diff   
experiment_id variation_id     model_id                                       
competition   default          deepseek_v3.2        10.9   8.4  1.7    18.5   
                               gpt_5                18.3   3.1 -1.1    -5.5   
                               gpt_5_mini           17.6   3.2  2.0    13.1   
                               gpt_5_nano            9.9   5.6  1.9    23.4   
                               qwen_3_max_thinking  17.5   0.0  0.4     2.4   
                               qwen_turbo           24.7   2.0 -0.3    -1.1   
              fully_

## Experiment 2. Choice

## Experiment 3. Familiarity & Choice